In [1]:
pip install transformers datasets torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import AutoModelForSeq2SeqLM, TrainingArguments, Trainer, AutoTokenizer
from datasets import Dataset
import torch
import pandas as pd

In [3]:
df = pd.read_csv('/kaggle/input/bank-full-csv/bank-full.csv', sep=';')
df.sample(5)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
42298,31,admin.,single,secondary,no,270,no,no,cellular,16,nov,176,1,200,2,success,no
35733,36,blue-collar,married,secondary,no,74,yes,no,cellular,8,may,168,2,-1,0,unknown,no
10041,35,blue-collar,single,secondary,no,406,yes,no,unknown,11,jun,126,12,-1,0,unknown,no
27932,45,services,married,secondary,no,-32,no,no,cellular,28,jan,56,2,-1,0,unknown,no
9008,52,unknown,married,primary,no,8251,no,no,unknown,5,jun,397,1,-1,0,unknown,no


In [5]:
df['y'].tail(10)

45201    yes
45202    yes
45203    yes
45204    yes
45205    yes
45206    yes
45207    yes
45208    yes
45209     no
45210     no
Name: y, dtype: object

In [6]:
# Преобразование данных в текст
def concatenate_text_bank(x):
    # Преобразуем бинарные значения в текст
    dft = 'has a credit in default' if x['default'] == 'yes' else 'has no credit in default'
    hsg = 'has housing loan' if x['housing'] == 'yes' else "doesn't have housing loan"
    ln = 'has a personal loan' if x['loan'] == 'yes' else 'has no personal loan'

    # Формируем текст описания
    full_text = (
        f"This customer is {x['age']} years old, "
        f"with {x['job']} occupation, "
        f"is {x['marital']}, "
        f"with {x['education']} level of education, "
        f"{dft}, "
        f"with average yearly balance of {x['balance']} euros, "
        f"{hsg}, "
        f"{ln}, "
        f"contacted via {x['contact']} type of contact, "
        f"on {x['day']} day of {x['month']}, "
        f"with the last call duration of {x['duration']} seconds, "
        f"{x['campaign']} times of call in current marketing campaign, "
        f"with {x['pdays']} days of pass between contacts, "
        f"{x['previous']} times of contacts in the previous campaign, "
        f"and last poutcome is {x['poutcome']}."
    )
    return full_text

In [7]:
df['text'] = df.apply(lambda x: concatenate_text_bank(x), axis=1)
df['text'].iloc[3]

'This customer is 47 years old, with blue-collar occupation, is married, with unknown level of education, has no credit in default, with average yearly balance of 1506 euros, has housing loan, has no personal loan, contacted via unknown type of contact, on 5 day of may, with the last call duration of 92 seconds, 1 times of call in current marketing campaign, with -1 days of pass between contacts, 0 times of contacts in the previous campaign, and last poutcome is unknown.'

In [8]:
# Преобразование в формат Dataset
dataset = Dataset.from_pandas(df[['text', 'y']])
dataset

Dataset({
    features: ['text', 'y'],
    num_rows: 45211
})

In [9]:
# Проверка доступности GPU
if torch.cuda.is_available():
    print("GPU доступен!")
    device = torch.device("cuda")  # Использовать GPU
else:
    print("GPU недоступен, используется CPU.")
    device = torch.device("cpu")  # Использовать CPU

GPU доступен!


In [10]:
# Загрузка модели и токенизатора
model = AutoModelForSeq2SeqLM.from_pretrained("bigscience/T0_3B")
tokenizer = AutoTokenizer.from_pretrained("bigscience/T0_3B")

# Перемещение модели на GPU
model.to(device)

config.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/11.4G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.86k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


T5ForConditionalGeneration(
  (shared): Embedding(32128, 2048)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 2048)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=2048, out_features=2048, bias=False)
              (k): Linear(in_features=2048, out_features=2048, bias=False)
              (v): Linear(in_features=2048, out_features=2048, bias=False)
              (o): Linear(in_features=2048, out_features=2048, bias=False)
              (relative_attention_bias): Embedding(32, 32)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=2048, out_features=5120, bias=False)
              (wi_1): Linear(in_features=2048, out_features=5120, bias=False)
       

In [11]:
def tokenize_function(examples):
    inputs = tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    labels = tokenizer(examples['y'], padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    inputs['labels'] = labels['input_ids']
    return inputs

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/45211 [00:00<?, ? examples/s]

In [12]:
tokenized_dataset

Dataset({
    features: ['text', 'y', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 45211
})

In [14]:
# Разделение данных на train и test
split_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)

train_data = split_dataset['train']
test_data = split_dataset['test']

In [15]:
# Удаляем ненужные колонки
train_data = train_data.remove_columns(['text', 'y'])
test_data = test_data.remove_columns(['text', 'y'])

In [16]:
train_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

In [17]:
# Проверка формы input_ids и attention_mask
print("Форма input_ids:", train_data[0]['input_ids'].shape)
print("Форма attention_mask:", train_data[0]['attention_mask'].shape)
print("Форма labels:", train_data[0]['labels'].shape)


Форма input_ids: torch.Size([128])
Форма attention_mask: torch.Size([128])
Форма labels: torch.Size([128])


In [18]:
train_data['attention_mask'][0]

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0])

In [19]:
train_data['input_ids'][0]

tensor([  100,   884,    19,  3538,   203,   625,     6,    28, 10473,     5,
        13792,     6,    19,   712,     6,    28,  6980,   593,    13,  1073,
            6,    65,   150,   998,    16,  4647,     6,    28,  1348,     3,
        22286,  2109,    13,     3,    18,  2668, 10186,     6,   744,    31,
           17,    43,  3499,  2289,     6,    65,   150,   525,  2289,     6,
            3, 12655,  1009,     3, 10791,   686,    13,   574,     6,    30,
         2838,   239,    13,     3,  7066,     6,    28,     8,   336,   580,
         8659,    13,     3, 15003,  3978,     6,   209,   648,    13,   580,
           16,   750,  1070,  2066,     6,    28,     3, 21594,   477,    13,
         1903,   344, 11463,     6,   305,   648,    13, 11463,    16,     8,
         1767,  2066,     6,    11,   336, 10964,    17,   287,    15,    19,
          119,     5,     1,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0])

In [20]:
train_data['labels'][0]

tensor([150,   1,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0])

In [21]:
# Заморозка всех слоев в энкодере
for param in model.encoder.parameters():
    param.requires_grad = False
    
# Заморозка всех слоев в декодере, кроме 23-го блока
for i, layer in enumerate(model.decoder.block):
    if i != 23:  # Указание на 23-й слой (индекс 23)
        for param in layer.parameters():
            param.requires_grad = False

# Проверка заморозки
for name, param in model.named_parameters():
    print(f"{name}: requires_grad = {param.requires_grad}")

shared.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.q.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.k.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.v.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.o.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.relative_attention_bias.weight: requires_grad = False
encoder.block.0.layer.0.layer_norm.weight: requires_grad = False
encoder.block.0.layer.1.DenseReluDense.wi_0.weight: requires_grad = False
encoder.block.0.layer.1.DenseReluDense.wi_1.weight: requires_grad = False
encoder.block.0.layer.1.DenseReluDense.wo.weight: requires_grad = False
encoder.block.0.layer.1.layer_norm.weight: requires_grad = False
encoder.block.1.layer.0.SelfAttention.q.weight: requires_grad = False
encoder.block.1.layer.0.SelfAttention.k.weight: requires_grad = False
encoder.block.1.layer.0.SelfAttention.v.weight: requires_grad = False
encoder.block.1.layer.0.SelfAtt

In [ ]:
torch.cuda.empty_cache()

In [23]:
training_args = TrainingArguments(
    output_dir='./results',
    run_name="my_t0_experiment",
    report_to="none",
    eval_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
)

In [24]:
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 30.12 MiB is free. Process 2490 has 14.71 GiB memory in use. Of the allocated memory 14.14 GiB is allocated by PyTorch, and 388.79 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)